<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
StateGraph Basics
</b></font> </br></p>

---

In [ ]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith Env-Vars VOR allen LangChain-Imports setzen
import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"]    = "M13-StateGraph-Basics"
os.environ["LANGSMITH_ENDPOINT"]   = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment, get_ipinfo, setup_api_keys,
    mprint, install_packages, mermaid, load_prompt
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

In [ ]:
#@title Pakete installieren{ display-mode: "form" }
# install_packages(["langchain-openai", "langgraph"])

# 1 | Übersicht
---



M12 hat gezeigt, **warum** LangGraph existiert und einen ersten 1-Node-Graphen gebaut.  
M13 geht tiefer: Wir bauen einen **vollständigen 2-Node-Graphen** mit mehreren State-Feldern,  
verschiedenen Edge-Typen und vollständiger Visualisierung.

**Was wir in M13 lernen**

| Thema | Inhalt |
|-------|--------|
| **State-Design** | TypedDict mit mehreren Feldern, Reducer vs. Überschreiben |
| **Node-Regeln** | Signatur, partieller Return, Fehlerbehandlung |
| **Edges** | `add_edge()`, kurze Vorschau auf `add_conditional_edges()` |
| **Kompilierung** | `compile()`, `recursion_limit`, Graph-Visualisierung |
| **Ausführung** | `invoke()`, `stream()`, State nach Ausführung inspizieren |
| **LangSmith** | Trace mit `run_name`, Node-by-Node-Sicht |

**Das Praxisbeispiel: Qualitäts-Agent**

Wir bauen einen **zweistufigen Analyse-Agenten**:

- **Node 1 `entwurf_node`**: Erstellt einen ersten Textentwurf
- **Node 2 `korrektorat_node`**: Verbessert den Entwurf

Jeder Node liest aus dem **gemeinsamen State** und schreibt seine Ergebnisse zurück.

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("openai:gpt-4o-mini", temperature=0.7)
print("LLM bereit:", llm.model_name)

In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  flowchart</font> </br></p>

diagram = '''
flowchart LR
    ST([START])
    N1["<b>Node 1</b>\nentwurf_node()\n▸ liest: anfrage\n▸ schreibt: entwurf"]
    N2["<b>Node 2</b>\nkorrektorat_node()\n▸ liest: entwurf\n▸ schreibt: finale_antwort"]
    FIN([FINISH])
    S[/"<b>State</b>\nanfrage: str\nentwurf: str\nfinale_antwort: str\nschritt: int"/]

    ST --> N1 --> N2 --> FIN
    N1 <-.->|"lesen / schreiben"| S
    N2 <-.->|"lesen / schreiben"| S

    style ST fill:#90EE90
    style FIN fill:#FFB6C1
    style S fill:#E8E8FF,stroke:#9999CC
'''

mermaid(diagram, width=900)

# 2 | StateGraph erstellen
---



**Der State: Einzige Wahrheitsquelle**

Der **State** ist das Herzstück jedes LangGraph-Workflows.  
Alle Nodes lesen daraus und schreiben dorthin – sie kommunizieren **nie direkt**.

**TypedDict: Schnell, typsicher, empfohlen**

| Kriterium | TypedDict | Pydantic BaseModel |
|-----------|----------|--------------------|
| **Empfehlung** | ✅ Für Graph-State | ⚠️ Nur für API-Grenzen |
| **Performance** | Kein Overhead | Langsamer (Validation) |
| **Typen** | Statische Hints | Strikte Laufzeit-Validation |

**Reducer: Wie der State aktualisiert wird**

LangGraph unterscheidet zwei Verhaltensweisen bei State-Updates:

| Typ | Verhalten | Beispiel |
|-----|-----------|----------|
| **Überschreiben** (Standard) | Neuer Wert ersetzt alten | `schritt: 2` → `schritt: 3` |
| **Reducer** (`add_messages`) | Neue Werte werden **angehängt** | `[msg1]` + `[msg2]` → `[msg1, msg2]` |

> **Wichtig:** Für `messages` immer `Annotated[list, add_messages]` verwenden –  
> sonst wird der Gesprächsverlauf bei jedem Node-Return überschrieben!

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages

# ─── State-Definition ────────────────────────────────────────────────────
class QualitaetsState(TypedDict):
    """State des Qualitaets-Agenten.

    Alle Felder werden von den Nodes geteilt.
    Jeder Node liest und schreibt nur die Felder, die er benoetigt.
    """
    messages:      Annotated[list, add_messages]  # Reducer: immer anhaengen
    anfrage:       str   # Eingabe des Nutzers (unveraendert)
    entwurf:       str   # Node 1 schreibt hier
    finale_antwort: str  # Node 2 schreibt hier
    schritt:       int   # Zaehlt Ausfuehrungsschritte

# ─── Initialer State ─────────────────────────────────────────────────────
start_state: QualitaetsState = {
    "messages":       [],
    "anfrage":        "",    # Wird beim Aufruf gesetzt
    "entwurf":        "",
    "finale_antwort": "",
    "schritt":        0,
}
print("State-Schema:")
for feld, wert in start_state.items():
    print(f"  {feld:18s}: {type(wert).__name__}")

In [ ]:
from langgraph.graph import StateGraph, START, END

# StateGraph mit dem State-Schema initialisieren
builder = StateGraph(QualitaetsState)

print("StateGraph erstellt")
print("Naechste Schritte: Nodes hinzufuegen, Edges verbinden, kompilieren")

# 3 | Nodes definieren
---



**Die 4 Regeln für Node-Funktionen**

| Regel | Beschreibung | Beispiel |
|-------|-------------|----------|
| **1. Signatur** | Empfängt den State, gibt `dict` zurück | `def node(state: State) -> dict:` |
| **2. Partieller Return** | Nur veränderte Felder zurückgeben | `return {"schritt": state["schritt"] + 1}` |
| **3. Kein `raise`** | Fehler als String im State speichern | `return {"fehler": str(e)}` |
| **4. Pure Funktion** | Kein globaler Seiteneffekt | State ist die einzige Kommunikation |

**Partieller Return – warum kein vollständiger State?**

LangGraph **merged** automatisch:

```python
# Aktueller State:
# { anfrage: "Hallo", entwurf: "", schritt: 0 }

# Node gibt zurück:
return {"entwurf": "Erster Entwurf...", "schritt": 1}

# Neuer State nach Merge:
# { anfrage: "Hallo", entwurf: "Erster Entwurf...", schritt: 1 }
```

> `anfrage` bleibt erhalten, obwohl der Node es nicht zurückgegeben hat!

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

entwurf_prompt = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m13_entwurf_prompt.md", mode="T")
korrektorat_prompt = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m13_korrektorat_prompt.md", mode="T")

# ─── Node 1: Entwurf erstellen ───────────────────────────────────────────
def entwurf_node(state: QualitaetsState) -> dict:
    """Erstellt einen ersten Textentwurf basierend auf der Anfrage.

    Liest: state["anfrage"]
    Schreibt: entwurf, schritt, messages
    """
    print(f"  [Node 1] Erstelle Entwurf fuer: '{state['anfrage'][:50]}...'")

    prompt_value = entwurf_prompt.invoke({"anfrage": state["anfrage"]})
    response = llm.invoke(prompt_value.messages)

    return {
        "messages": [*prompt_value.messages, response],
        "entwurf": response.content,
        "schritt": state["schritt"] + 1,
    }

print("Node 1 (entwurf_node) definiert")


In [ ]:
# ─── Node 2: Korrektorat durchfuehren ────────────────────────────────────
def korrektorat_node(state: QualitaetsState) -> dict:
    """Verbessert den Entwurf aus Node 1.

    Liest: state["entwurf"]
    Schreibt: finale_antwort, schritt, messages
    """
    print(f"  [Node 2] Verbessere Entwurf ({len(state['entwurf'])} Zeichen)")

    prompt_value = korrektorat_prompt.invoke({"entwurf": state["entwurf"]})
    response = llm.invoke(prompt_value.messages)

    return {
        "messages": [*prompt_value.messages, response],
        "finale_antwort": response.content,
        "schritt": state["schritt"] + 1,
    }

print("Node 2 (korrektorat_node) definiert")


In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  flowchart</font> </br></p>

diagram = '''
flowchart TB
    subgraph Node1["entwurf_node() – Node 1"]
        N1R["Liest: anfrage"] --> N1L["LLM: Erstellt Entwurf"]
        N1L --> N1W["Schreibt: entwurf, schritt"]
    end

    subgraph State["Gemeinsamer State"]
        direction LR
        F1["anfrage: str"]
        F2["entwurf: str"]
        F3["finale_antwort: str"]
        F4["schritt: int"]
        F5["messages: list"]
    end

    subgraph Node2["korrektorat_node() – Node 2"]
        N2R["Liest: entwurf"] --> N2L["LLM: Verbessert Text"]
        N2L --> N2W["Schreibt: finale_antwort, schritt"]
    end

    Node1 <-->|merge| State
    Node2 <-->|merge| State

    style State fill:#E8E8FF,stroke:#9999CC
    style Node1 fill:#E8FFE8
    style Node2 fill:#FFE8E8
'''

mermaid(diagram, width=1050)

# 4 | Edges verbinden
---



**Edge-Typen**

| Typ | Methode | Wann verwenden |
|-----|---------|----------------|
| **Einfache Edge** | `add_edge(from, to)` | Immer dieser Weg |
| **Conditional Edge** | `add_conditional_edges(from, fn)` | Entscheidung zur Laufzeit |

**START und END**

```python
from langgraph.graph import START, END

# START = spezieller Eingangsknoten (kein Node, kein Code)
# END   = spezieller Ausgangsknoten (Workflow beenden)
builder.add_edge(START, "mein_node")   # Einstieg
builder.add_edge("mein_node", END)     # Ausstieg
```

**Conditional Edges – kurze Vorschau (Details in M14)**

```python
# Routing-Funktion: gibt den Namen des nächsten Nodes zurück
def routing_fn(state: State) -> str:
    if state["qualitaet"] == "schlecht":
        return "nochmal_verbessern"   # Schleife
    return END                         # Fertig

builder.add_conditional_edges(
    "korrektorat",    # Von diesem Node
    routing_fn,       # Diese Funktion entscheidet
)
```

> **M14** zeigt Conditional Edges und Tool-Loops im Detail.

In [ ]:
# Nodes zum Builder hinzufügen
builder.add_node("entwurf",    entwurf_node)
builder.add_node("korrektorat", korrektorat_node)

# Edges verbinden: START → entwurf → korrektorat → END
builder.add_edge(START,         "entwurf")
builder.add_edge("entwurf",     "korrektorat")
builder.add_edge("korrektorat", END)

print("Nodes registriert:", list(builder.nodes.keys()))
print("Graph-Struktur:")
print("  START → entwurf → korrektorat → END")

# 5 | Graph kompilieren & testen
---



**Kompilierung: Builder → unveränderlicher Graph**

Nach `compile()` kann der Graph **nicht mehr verändert** werden.  
Die Kompilierung prüft den Graphen auf Vollständigkeit und erzeugt den ausführbaren Workflow.

| Option | Beschreibung | Standard |
|--------|-------------|----------|
| `checkpointer=` | Persistenz (MemorySaver/PostgresSaver) | Kein Checkpointing |
| `interrupt_before=` | Nodes bei denen pausiert wird | `[]` |
| `interrupt_after=` | Nodes nach denen pausiert wird | `[]` |

> **Best Practice:** Immer `draw_mermaid_png()` nach `compile()` aufrufen –  
> zeigt den **tatsächlich** kompilierten Graphen.

In [ ]:
from IPython.display import Image, display

# Graph kompilieren
graph = builder.compile()
print("Graph kompiliert")

# Visualisierung: Pflicht nach compile()
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"Visualisierung nicht verfuegbar: {e}")
    print(graph.get_graph().draw_mermaid())

In [ ]:
# Graph ausführen mit invoke()
initial_state: QualitaetsState = {
    "messages":       [],
    "anfrage":        "Die Bedeutung von RAG-Systemen in modernen KI-Anwendungen",
    "entwurf":        "",
    "finale_antwort": "",
    "schritt":        0,
}

print("Starte Graphen...\n")
ergebnis = graph.invoke(initial_state)

mprint(f"""
## Ergebnis nach invoke()

**Schritte ausgefuehrt:** {ergebnis["schritt"]}

**Entwurf (Node 1):**
{ergebnis["entwurf"]}

---

**Finale Antwort (Node 2):**
{ergebnis["finale_antwort"]}

**Nachrichten im State:** {len(ergebnis["messages"])} Eintraege
""")

In [ ]:
# State nach Ausführung inspizieren
print("=== State-Inspektion ===")
for feld, wert in ergebnis.items():
    if feld == "messages":
        print(f"  messages: {len(wert)} Eintraege")
        for i, msg in enumerate(wert):
            typ = type(msg).__name__
            inhalt = msg.content[:60].replace("\n", " ") + "..."
            print(f"    [{i}] {typ}: {inhalt}")
    elif isinstance(wert, str) and len(wert) > 60:
        print(f"  {feld}: {wert[:60]}...")
    else:
        print(f"  {feld}: {repr(wert)}")

In [ ]:
# Graph mit stream() Schritt für Schritt verfolgen
print("=== stream() – Node-by-Node ===\n")

stream_state: QualitaetsState = {
    **initial_state,
    "anfrage": "Warum ist Checkpointing in LangGraph wichtig?",
}

for schritt, event in enumerate(graph.stream(stream_state, stream_mode="updates")):
    node_name = list(event.keys())[0]
    node_output = event[node_name]

    print(f"--- Schritt {schritt + 1}: Node '{node_name}' ---")
    for feld, wert in node_output.items():
        if feld == "messages":
            print(f"  messages: +{len(wert)} neue Eintraege")
        elif isinstance(wert, str) and len(wert) > 80:
            print(f"  {feld}: {wert[:80]}...")
        else:
            print(f"  {feld}: {repr(wert)}")
    print()

# 6 | Graph im LangSmith
---



LangGraph-Traces in LangSmith zeigen eine **Node-by-Node-Sicht** des Workflows –
ideal zum Verstehen und Debuggen.

**Was LangSmith für StateGraphen zeigt**

| Ebene | Inhalt | Nutzen |
|-------|--------|--------|
| **Graph-Ebene** | Gesamtlaufzeit, Input/Output | Überblick |
| **Node-Ebene** | Ein Eintrag pro Node | Welcher Node lief wie lang? |
| **LLM-Ebene** | Prompt, Response, Tokens | Prompt-Debugging |
| **State-Diff** | State vor/nach jedem Node | Was hat sich verändert? |

**run_name und Tags**

```python
config = {
    "run_name": "Qualitaets-Agent",
    "tags":     ["m13", "zwei-node-graph"],
}
graph.invoke(state, config=config)
```

In [ ]:
# Ausführung mit LangSmith-Konfiguration
langsmith_config = {
    "run_name": "Qualitaets-Agent-Demo",
    "tags":     ["m13", "zwei-node-graph", "invoke"],
    "metadata": {"modul": "M13", "version": "1.0"},
}

trace_state: QualitaetsState = {
    "messages":       [],
    "anfrage":        "Erklaere den Unterschied zwischen invoke() und stream() in LangGraph",
    "entwurf":        "",
    "finale_antwort": "",
    "schritt":        0,
}

print("Starte Trace...\n")
trace_result = graph.invoke(trace_state, config=langsmith_config)

mprint(f"""
## LangSmith-Trace

**Projekt:** M13-StateGraph-Basics
**Run-Name:** Qualitaets-Agent-Demo
**Schritte:** {trace_result["schritt"]}

**Finale Antwort:**
{trace_result["finale_antwort"][:400]}...

> Trace in LangSmith unter Projekt **M13-StateGraph-Basics** einsehen.
""")

# A | Aufgabe
---



Die Aufgabestellungen unten bieten Anregungen, Sie können aber auch gerne eine andere Herausforderung angehen.

<p><font color='black' size="5">
Drei-Node-Übersetzungs-Agent
</font></p>

Bauen Sie einen StateGraph mit **drei Nodes** für einen Übersetzungs-Workflow:

**State:**
```python
class UebersetzungsState(TypedDict):
    messages:     Annotated[list, add_messages]
    originaltext: str     # Eingabe
    uebersetzung: str     # Node 1 schreibt
    rueckuebersetzung: str # Node 2 schreibt (DE→EN→DE)
    qualitaet: str         # Node 3 bewertet
    schritt: int
```

**Teilaufgaben:**
1. Definieren Sie die 3 Nodes (Übersetzen EN→DE, Rück-Übersetzen DE→EN, Qualität bewerten)
2. Verbinden Sie die Nodes sequentiell: START → node1 → node2 → node3 → END
3. Kompilieren Sie den Graphen und visualisieren Sie ihn mit `draw_mermaid_png()`
4. Führen Sie einen Test mit einem englischen Satz durch
5. Vergleichen Sie Original und Rück-Übersetzung (wie viel geht verloren?)

**Bonus:** Verwenden Sie `stream()` statt `invoke()`, um jeden Schritt einzeln zu sehen,
und analysieren Sie den Trace in LangSmith.